In [ ]:
from pathlib import Path
import os, json, hashlib, platform
import numpy as np
import pandas as pd

PROJECT = Path.cwd().resolve()
ROOT = Path(os.environ.get("PINPAD_DATA_ROOT", "/home/cse/pinpad/myenv/GNURadio_1/Correlation_Responses/OneSubcarrierLowDataRate"))
OUTPUT = Path(os.environ.get("PINPAD_OUTPUT_ROOT", "/home/cse/pinpad/myenv/GNURadio_1/PINPAD_Compact_Results"))
N_BUTTONS, N_PER_BUTTON = 12, 200
RESPONSE_LENGTH = 3624
INPUT_FORMAT = "magnitude"
NORMALIZATION = "source_global_zscore"
DRIFT_MODE = "none"
RUN_SEEDS = [42,43,44,45,46]
SOURCE = [(c,"R1") for c in range(1,6)]
ADAPT_CC = [(c,"R1") for c in range(6,11)]
ADAPT_CR = [(c,"R2") for c in range(1,6)]
ADAPT_CCR = [(c,"R2") for c in range(11,16)]
ADAPT = {"Source":SOURCE,"CC":ADAPT_CC,"CR":ADAPT_CR,"CCR":ADAPT_CCR}
EXPECTED = set(sum(ADAPT.values(),[]))
COLLECTION_MAP = ROOT/"collection_paths.csv"


def collection_paths():
    mapping = {}
    if COLLECTION_MAP.is_file():
        table = pd.read_csv(COLLECTION_MAP,dtype=str,keep_default_na=False)
        if not {"card","reader","directory"} <= set(table):
            raise ValueError("collection_paths.csv requires card,reader,directory")
        for row in table.itertuples(index=False):
            key = (int(row.card),row.reader.strip().upper())
            if key in mapping or not row.directory.strip():
                raise ValueError(f"Duplicate or empty collection mapping: {key}")
            path = Path(row.directory).expanduser()
            mapping[key] = (path if path.is_absolute() else ROOT/path).resolve()
    else:
        for card,reader in sorted(EXPECTED):
            candidates = {p.resolve() for p in (ROOT/reader/f"card{card}",
                ROOT/reader.lower()/f"card{card}",ROOT/f"card{card}"/reader,
                ROOT/f"card{card}"/reader.lower()) if p.is_dir()}
            if len(candidates)>1: raise ValueError(f"Ambiguous physical collection: {card}, {reader}")
            if candidates: mapping[(card,reader)] = candidates.pop()
    missing = [key for key in sorted(EXPECTED) if key not in mapping or not mapping[key].is_dir()]
    if missing:
        raise FileNotFoundError(f"Missing collections {missing}. Configure PINPAD_DATA_ROOT and collection_paths.csv; see README.")
    paths = [mapping[k] for k in sorted(EXPECTED)]
    if len(set(paths)) != len(paths): raise ValueError("Different physical collections map to the same directory")
    return mapping


def read_collection(key,mapping):
    parts = []
    for button in range(N_BUTTONS):
        hits = [p for p in (mapping[key]/f"button{button}.npy",mapping[key]/f"Button{button}.npy") if p.is_file()]
        if len(hits)!=1: raise FileNotFoundError(f"Expected one button{button}.npy in {mapping[key]}")
        x = np.load(hits[0],allow_pickle=False)
        if x.shape!=(N_PER_BUTTON,RESPONSE_LENGTH) or not np.iscomplexobj(x) or not np.isfinite(x).all():
            raise ValueError(f"Expected finite complex ({N_PER_BUTTON},{RESPONSE_LENGTH}) array: {hits[0]}")
        parts.append(x)
    return np.concatenate(parts),np.repeat(np.arange(N_BUTTONS),N_PER_BUTTON)


def load_group(keys,mapping):
    values = [read_collection(k,mapping) for k in keys]
    return (np.concatenate([x for x,y in values]),np.concatenate([y for x,y in values]),
            np.concatenate([np.full(len(y),k[0]) for k,(x,y) in zip(keys,values)]))


def features(raw):
    x = np.asarray(raw)
    if x.ndim!=2 or x.shape[1]!=RESPONSE_LENGTH or not np.iscomplexobj(x) or not np.isfinite(x).all():
        raise ValueError("Expected finite complex response segments")
    if DRIFT_MODE=="linear_complex":
        t = np.linspace(-1.,1.,x.shape[1])
        x = x-x.mean(axis=1,keepdims=True)
        slope = (x@t)/np.dot(t,t)
        x = x-slope[:,None]*t
    elif DRIFT_MODE!="none": raise ValueError("DRIFT_MODE must be none or linear_complex")
    if INPUT_FORMAT!="magnitude": raise ValueError("This release uses magnitude inputs")
    return np.abs(x).astype(np.float32)[...,None]


def normalize(x,stats):
    return ((x-np.asarray(stats['mean'],dtype=np.float32))/np.asarray(stats['std'],dtype=np.float32)).astype(np.float32)


def load_splits():
    f = pd.read_csv(SPLITS_CSV,dtype=str,keep_default_na=False)
    if not {"run_id","card","reader","partition"} <= set(f): raise ValueError("Invalid card split columns")
    f['card'] = f.card.astype(int)
    f['reader'] = f.reader.str.upper().str.strip()
    if set(f.run_id)!={f"run{i}" for i in range(1,6)}: raise ValueError("Provide exactly run1–run5")
    for run_id,g in f.groupby('run_id'):
        if g.duplicated(['card','reader']).any() or set(zip(g.card,g.reader))!=EXPECTED:
            raise ValueError(f"{run_id}: assign every development collection exactly once")
        if not set(g.partition)<={'train','validation'}: raise ValueError("Invalid development partition")
        source = g.reader.eq('R1') & g.card.between(1,5)
        if set(g.loc[source,'partition'])!={'train','validation'}: raise ValueError("Source needs train and validation cards")
        if not g.loc[~source,'partition'].eq('train').all(): raise ValueError("Targets must be unlabeled adaptation training collections")
    return f


def response_hashes(raw):
    return np.asarray([hashlib.sha256(np.ascontiguousarray(x,dtype='<c16').tobytes()).hexdigest() for x in raw])


def prepare_data(run_id,splits):
    mapping = collection_paths()
    raw,y,cards = load_group(SOURCE,mapping)
    selected = splits[splits.run_id.eq(run_id)]
    source = selected[selected.reader.eq('R1') & selected.card.between(1,5)]
    validation_cards = set(source.loc[source.partition.eq('validation'),'card'])
    val = np.isin(cards,list(validation_cards)); train = ~val
    h = response_hashes(raw)
    if set(h[train]) & set(h[val]): raise ValueError("Identical source captures cross training and validation")
    x = features(raw)
    stats = dict(mean=x[train].mean(axis=(0,1),dtype=np.float64).astype(np.float32).tolist(),
        std=np.maximum(x[train].std(axis=(0,1),dtype=np.float64).astype(np.float32),1e-7).tolist())
    x = normalize(x,stats)
    targets = {}
    audit = [pd.DataFrame(dict(domain='Source',card=cards,partition=np.where(val,'validation','train'),response_sha256=h))]
    for domain,keys in (("CC",ADAPT_CC),("CR",ADAPT_CR),("CCR",ADAPT_CCR)):
        target_raw,unused,tcards = load_group(keys,mapping)
        th = response_hashes(target_raw)
        if set(th)&set(h[val]): raise ValueError(f"{domain}: adaptation captures overlap source validation")
        targets[domain] = normalize(features(target_raw),stats)
        audit.append(pd.DataFrame(dict(domain=domain,card=tcards,partition='train',response_sha256=th)))
    audit = pd.concat(audit,ignore_index=True)
    prep = dict(input_format=INPUT_FORMAT,signal_length=RESPONSE_LENGTH,normalization=NORMALIZATION,
        source_stats=stats,drift_mode=DRIFT_MODE,classes=[f"B{i}" for i in range(12)])
    return x[train],y[train],x[val],y[val],targets,prep,audit


def save_json(path,data):
    Path(path).write_text(json.dumps(data,indent=2)+"\n")


def save_audit(run,selected,audit):
    selected.to_csv(run/"card_split.csv",index=False)
    audit.to_csv(run/"capture_hashes.csv",index=False)
    duplicates = audit[audit.response_sha256.duplicated(keep=False)]
    duplicates.to_csv(run/"duplicate_response_audit.csv",index=False)
    if not duplicates.empty: print("Repeated response hashes recorded in duplicate_response_audit.csv")


In [ ]:
os.environ.setdefault("TF_DETERMINISTIC_OPS","1")
import tensorflow as tf
from tensorflow import keras

BATCH_SIZE = 48
LEARNING_RATE = 0.001
ENCODER_DROPOUT = 0.3
CLASSIFIER_DROPOUT = 0.3
CORAL_WEIGHT = 1.0
SOURCE_EPOCHS, ADAPT_EPOCHS = 30, 30
SOURCE_PATIENCE, ADAPT_PATIENCE = 7, 7
POOL_SIZES = [2,2]
POOL_STRIDES = [2,2]
CONV_PADDING = "same"
CONV_STRIDES = [1,1]
CHECKPOINT_RULE = "best_source_validation"
ADAM_KWARGS = {}

LR_PATIENCE = 3
LR_FACTOR = 0.5
MIN_LR = 1e-6
GRADIENT_CLIP_NORM = 5.0


def check_dl_settings():
    required = dict(batch_size=BATCH_SIZE,learning_rate=LEARNING_RATE,encoder_dropout=ENCODER_DROPOUT,classifier_dropout=CLASSIFIER_DROPOUT,
        coral_weight=CORAL_WEIGHT,source_epochs=SOURCE_EPOCHS,adapt_epochs=ADAPT_EPOCHS,
        pool_sizes=POOL_SIZES,pool_strides=POOL_STRIDES,padding=CONV_PADDING,
        conv_strides=CONV_STRIDES,checkpoint_rule=CHECKPOINT_RULE,adam_kwargs=ADAM_KWARGS)
    missing = [k for k,v in required.items() if v is None]
    if missing: raise ValueError(f"Supply original DL settings: {missing}")
    if BATCH_SIZE < 12 or BATCH_SIZE % 12: raise ValueError("Batch size must be divisible by 12")
    if BATCH_SIZE < 2 or SOURCE_EPOCHS < 1 or ADAPT_EPOCHS < 1:
        raise ValueError("Batch size must be >=2; epoch counts must be positive")
    if not (0 <= ENCODER_DROPOUT < 1 and 0 <= CLASSIFIER_DROPOUT < 1) or LEARNING_RATE <= 0 or CORAL_WEIGHT < 0:
        raise ValueError("Invalid training hyperparameters")
    if CHECKPOINT_RULE not in ('best_source_validation','last_epoch'):
        raise ValueError("Set the original checkpoint selection rule")

SPLITS_CSV = Path(os.environ.get("PINPAD_DL_SPLITS",PROJECT/"manifests/dl_card_splits.csv"))


In [ ]:
def build_model():
    channels = 1 if INPUT_FORMAT == "magnitude" else 2
    inputs = keras.Input((RESPONSE_LENGTH,channels))
    x = inputs
    for i,filters in enumerate((32,64)):
        x = keras.layers.Conv1D(filters,7,padding=CONV_PADDING,strides=CONV_STRIDES[i])(x)
        x = keras.layers.BatchNormalization()(x)
        x = keras.layers.ReLU()(x)
        x = keras.layers.MaxPooling1D(POOL_SIZES[i],strides=POOL_STRIDES[i])(x)
    x = keras.layers.LSTM(64,return_sequences=True)(x)
    x = keras.layers.GlobalAveragePooling1D()(x)
    x = keras.layers.Dense(64,activation="relu")(x)
    x = keras.layers.Dropout(ENCODER_DROPOUT)(x)
    encoder = keras.Model(inputs,x,name="encoder")
    embedding = keras.Input((64,))
    z = keras.layers.Dense(64,activation="relu")(embedding)
    z = keras.layers.BatchNormalization()(z)
    z = keras.layers.Dropout(CLASSIFIER_DROPOUT)(z)
    logits = keras.layers.Dense(12)(z)
    classifier = keras.Model(embedding,logits,name="classifier")
    response = keras.Input((RESPONSE_LENGTH,channels))
    model = keras.Model(response,classifier(encoder(response)))
    return model, encoder, classifier


def covariance(x):
    x = x-tf.reduce_mean(x,axis=0,keepdims=True)
    return tf.matmul(x,x,transpose_a=True)/tf.cast(tf.shape(x)[0]-1,x.dtype)


def coral(source,target):
    dimension = tf.cast(tf.shape(source)[1],source.dtype)
    return tf.reduce_sum(tf.square(covariance(source)-covariance(target)))/(4*dimension**2)


def balanced_source_batches(x,y,batch,rng):
    per_class = batch//12
    pools = {c:rng.permutation(np.flatnonzero(y==c)) for c in range(12)}
    steps = min(len(ix)//per_class for ix in pools.values())
    for step in range(steps):
        indices = np.concatenate([pools[c][step*per_class:(step+1)*per_class] for c in range(12)])
        rng.shuffle(indices)
        yield x[indices],y[indices]


def fit_stage(model,encoder,classifier,X,y,Xv,yv,targets=None,epochs=30,patience=7):
    if targets is not None and set(targets) != {"CC","CR","CCR"}:
        raise ValueError("Supply separate CC, CR, and CCR training arrays")
    if min([len(X)]+[len(v) for v in (targets or {}).values()]) < BATCH_SIZE:
        raise ValueError("Each domain must contain at least one full batch")
    if any(np.sum(y==c) < BATCH_SIZE//12 for c in range(12)):
        raise ValueError("Source training needs every class in each balanced batch")
    tf.keras.utils.set_random_seed(SEED)
    optimizer = keras.optimizers.Adam(learning_rate=LEARNING_RATE,**ADAM_KWARGS)
    loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    best_accuracy,best_weights,best_epoch,wait,lr_wait,history = -np.inf,None,0,0,0,[]
    target_rng = {k:np.random.default_rng(SEED+1000+i) for i,k in enumerate((targets or {}))}
    for epoch in range(1,epochs+1):
        batches = list(balanced_source_batches(X,y,BATCH_SIZE,np.random.default_rng(SEED+epoch)))
        needed = len(batches)*BATCH_SIZE
        target_order = {}
        for k,v in (targets or {}).items():
            order = target_rng[k].permutation(len(v))
            target_order[k] = np.resize(order,needed) if len(order)<needed else order
        losses,correct,seen = [],0,0
        for step,(xb,yb) in enumerate(batches):
            start,stop = step*BATCH_SIZE,(step+1)*BATCH_SIZE
            with tf.GradientTape() as tape:
                zs = encoder(xb,training=True)
                logits = classifier(zs,training=True)
                ce = loss_fn(yb,logits)
                terms = []
                for k in ("CC","CR","CCR") if targets is not None else ():
                    xt = targets[k][target_order[k][start:stop]]
                    tf.debugging.assert_equal(tf.shape(xb)[0],tf.shape(xt)[0])
                    terms.append(coral(zs,encoder(xt,training=True)))
                alignment = tf.add_n(terms)/3.0 if terms else tf.constant(0.,dtype=ce.dtype)
                loss = ce+CORAL_WEIGHT*alignment
                tf.debugging.assert_all_finite(loss,"Nonfinite loss")
            variables = encoder.trainable_variables+classifier.trainable_variables
            gradients = tape.gradient(loss,variables)
            pairs = [(g,v) for g,v in zip(gradients,variables) if g is not None]
            clipped,_ = tf.clip_by_global_norm([g for g,v in pairs],GRADIENT_CLIP_NORM)
            for g in clipped: tf.debugging.assert_all_finite(g,"Nonfinite gradient")
            optimizer.apply_gradients(zip(clipped,[v for g,v in pairs]))
            losses.append([float(loss),float(ce),float(alignment)]+[float(t) for t in terms])
            correct += int(np.sum(logits.numpy().argmax(1)==yb))
            seen += len(yb)
        val_logits = model.predict(Xv,batch_size=128,verbose=0)
        accuracy = float(np.mean(val_logits.argmax(1)==yv))
        avg = np.mean(losses,axis=0)
        record = dict(epoch=epoch,loss=float(avg[0]),cross_entropy=float(avg[1]),
            coral=float(avg[2]),train_accuracy=correct/seen,
            validation_loss=float(loss_fn(yv,val_logits)),validation_accuracy=accuracy,
            learning_rate=float(optimizer.learning_rate.numpy()))
        if targets is not None:
            record.update({f"coral_{k}":float(avg[3+i]) for i,k in enumerate(("CC","CR","CCR"))})
        history.append(record)
        print(f"Epoch {epoch}: loss={avg[0]:.6f}, validation={accuracy:.4%}")
        if accuracy > best_accuracy+1e-8:
            best_accuracy,best_weights,best_epoch,wait,lr_wait = accuracy,model.get_weights(),epoch,0,0
        else:
            wait,lr_wait = wait+1,lr_wait+1
            if lr_wait >= LR_PATIENCE:
                optimizer.learning_rate.assign(max(float(optimizer.learning_rate.numpy())*LR_FACTOR,MIN_LR))
                lr_wait = 0
            if patience is not None and wait >= patience: break
    if best_weights is None: raise RuntimeError("No checkpoint selected")
    if CHECKPOINT_RULE == "best_source_validation": model.set_weights(best_weights)
    return pd.DataFrame(history),dict(best_epoch=best_epoch,best_source_validation_accuracy=best_accuracy,
        checkpoint_rule=CHECKPOINT_RULE)


In [ ]:
check_dl_settings()
splits = load_splits()
for index in range(5):
    run_id,SEED = f"run{index+1}",RUN_SEEDS[index]
    RUN = OUTPUT / "DL" / run_id
    if RUN.exists(): raise FileExistsError(f"Choose a new output root: {RUN}")
    keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    tf.config.experimental.enable_op_determinism()
    X,y,Xv,yv,targets,preprocessing,audit = prepare_data(run_id,splits)
    RUN.mkdir(parents=True)
    save_audit(RUN,splits[splits.run_id.eq(run_id)],audit)
    save_json(RUN/"preprocessing.json",preprocessing)
    save_json(RUN/"training_config.json",dict(seed=SEED,batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,encoder_dropout=ENCODER_DROPOUT,classifier_dropout=CLASSIFIER_DROPOUT,coral_weight=CORAL_WEIGHT,
        source_epochs=SOURCE_EPOCHS,adapt_epochs=ADAPT_EPOCHS,
        source_patience=SOURCE_PATIENCE,adapt_patience=ADAPT_PATIENCE,
        pool_sizes=POOL_SIZES,pool_strides=POOL_STRIDES,conv_padding=CONV_PADDING,
        conv_strides=CONV_STRIDES,checkpoint_rule=CHECKPOINT_RULE,adam_kwargs=ADAM_KWARGS,
        lr_patience=LR_PATIENCE,lr_factor=LR_FACTOR,min_lr=MIN_LR,gradient_clip_norm=GRADIENT_CLIP_NORM,
        balanced_source=True,target_batching="separate equal-size CC/CR/CCR",
        source_pretraining="separate stage; not present in supplied joint-training notebook",
        protocol="reference DL five held-out source cards; not original manuscript assignments",
        python=platform.python_version(),keras=keras.__version__,tensorflow=tf.__version__,numpy=np.__version__,pandas=pd.__version__))
    model,encoder,classifier = build_model()
    history,selection = fit_stage(model,encoder,classifier,X,y,Xv,yv,epochs=SOURCE_EPOCHS,patience=SOURCE_PATIENCE)
    model.save(RUN/"source.keras")
    save_json(RUN/"source_selection.json",selection)
    history.to_csv(RUN/"source_history.csv",index=False)
    model.trainable = True
    history,selection = fit_stage(model,encoder,classifier,X,y,Xv,yv,targets=targets,
                        epochs=ADAPT_EPOCHS,patience=ADAPT_PATIENCE)
    model.save(RUN/"adapted_deepcoral.keras")
    save_json(RUN/"adaptation_selection.json",selection)
    history.to_csv(RUN/"adaptation_history.csv",index=False)
    print("Saved:",RUN)


In [ ]:
def preprocess_new_responses(raw,prep):
    x = np.asarray(raw)
    if x.ndim==1: x=x[None,:]
    if x.ndim!=2 or x.shape[1]!=prep['signal_length'] or not np.iscomplexobj(x) or not np.isfinite(x).all():
        raise ValueError("Expected finite complex responses (N,T)")
    if prep['drift_mode']=='linear_complex':
        t=np.linspace(-1.,1.,x.shape[1]); x=x-x.mean(axis=1,keepdims=True)
        x=x-((x@t)/np.dot(t,t))[:,None]*t
    elif prep['drift_mode']!='none': raise ValueError("Unsupported saved drift mode")
    mag=np.abs(x).astype(np.float32)[...,None]
    stats=prep['source_stats']
    return ((mag-np.asarray(stats['mean'],dtype=np.float32))/np.asarray(stats['std'],dtype=np.float32)).astype(np.float32)


def load_dl_predictor(run_directory):
    run_directory=Path(run_directory)
    saved=keras.models.load_model(run_directory/"adapted_deepcoral.keras",compile=False)
    prep=json.loads((run_directory/"preprocessing.json").read_text())
    def predict(raw):
        x=preprocess_new_responses(raw,prep)
        logits=saved.predict(x,batch_size=128,verbose=0)
        return tf.nn.softmax(logits,axis=-1).numpy()
    return predict
